In [21]:
import csv
import re
import pandas as pd
import math

In [2]:
def read_sales(path):
    
    with open(path, 'r') as f:
        data = csv.DictReader(f)
        rows = list(data)

    # print(f'Number of Rows: {len(rows)}')
    # print(f'Columns: {data.fieldnames}')
    return rows

In [3]:
read_sales('./data/electronics_sales.csv')

[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [32]:
# Convert numeric data types
def parse_number(value):

    number_words = {
        'one': 1.0,
        'two': 2.0,
        'three': 3.0
    }

    if value is None:
        raise ValueError("Missing Value")
    
    if isinstance(value, (int, float)):
        return value
    
    value = str(value).strip()

    if value.lower() in {'', 'n/a', 'na', 'nan', 'none', 'null', '-'}:
        return None
    if value.lower() in number_words:
        return number_words[value.lower()]
    
    if not value:
        raise ValueError("Empty Value")
    
    value = re.sub(r"^KES\s*", "", value, flags=re.IGNORECASE)
    
    number = float(value.replace(',', ''))
    
    if number < 0:
        number = -number

    return number

In [5]:
def convert_rows(rows):
    numeric_cols = ['quantity', 'unit_price_kes']

    repaired_rows = []
    skipped_rows = []
    

    for index, old_row in enumerate(rows, start=1):
        row = old_row.copy()
        errors = []

        for col in row:
            if col not in numeric_cols:
                continue
            try:
                row[col] = parse_number(row[col])
            except ValueError as error:
                errors.append(f'{col}: {error}')
        if errors:
            skipped_rows.append({
                'index': index,
                'row': old_row,
                'reason': '; '.join(errors)
            })
        else:
            repaired_rows.append(row)
    return repaired_rows, skipped_rows

In [6]:
clean_rows, skipped_rows = convert_rows(read_sales('./data/electronics_sales.csv'))

print('Clean rows:', len(clean_rows))
print('Skipped rows:', len(skipped_rows))


Clean rows: 47
Skipped rows: 3


In [7]:
# Calculate total revenue
def total_revenue(rows):
    total_revenue = 0

    for row in rows:
        quantity = float(row['quantity'])
        unit_price = float(row['unit_price_kes'])

        total_revenue += quantity * unit_price

    formatted_revenue = f'KES {total_revenue:,.2f}'
    return formatted_revenue

In [8]:
total_revenue(clean_rows)

'KES 1,317,210.00'

In [9]:
def revenue_by(rows, field):
    revenue = {}

    for row in rows:
        category = row[field]
        quantity = float(row['quantity'])
        unit_price = float(row['unit_price_kes'])

        row_revenue = quantity * unit_price

        if category not in revenue:
            revenue[category] = 0 

        revenue[category] += row_revenue

    return revenue

In [10]:
product_revenue = revenue_by(clean_rows, 'product')
store_revenue = revenue_by(clean_rows, 'store')
payment_method_revenue = revenue_by(clean_rows, 'payment_method')

print('Product Revenue')
for product, revenue in sorted(product_revenue.items(), key=lambda prod: prod[1], reverse=True):
    print(f'{product}: KES {revenue:,.2f}')

print('\nStore Revenue')
for store, revenue in sorted(store_revenue.items(), key=lambda prod: prod[1], reverse=True):
    print(f'{store}: KES {revenue:,.2f}')

print('\nPayment Methods Revenue')
for payment, revenue in sorted(payment_method_revenue.items(), key=lambda prod: prod[1], reverse=True):
    print(f'{payment}: KES {revenue:,.2f}')

Product Revenue
HP 15 Laptop: KES 389,994.00
Hisense 43in Smart TV: KES 272,993.00
Lenovo Tab M10: KES 199,992.00
Tecno Spark 20: KES 161,988.00
Infinix Hot 40: KES 63,996.00
Samsung Galaxy A15: KES 56,997.00
JBL Go 3 Speaker: KES 49,491.00
Anker Power Bank 20000: KES 38,691.00
Xiaomi Redmi Buds: KES 31,990.00
Oraimo FreePods: KES 19,593.00
USB-C Charger 25W: KES 19,487.00
Safaricom Neon Ray: KES 11,998.00

Store Revenue
Kisumu Mega Plaza: KES 449,678.00
Nairobi CBD: KES 337,483.00
Westgate: KES 235,979.00
Mombasa City Mall: KES 154,082.00
Nakuru Westside: KES 139,988.00

Payment Methods Revenue
M-Pesa: KES 738,568.00
Cash: KES 310,773.00
Card: KES 267,869.00


In [11]:
# Rank the top three products
def top_n(totals, n):
    top_n_products = sorted(totals.items(), key=lambda prod: prod[1], reverse=True)[:n]
    for product, revenue in top_n_products:
        print(product)


In [12]:
top_n(product_revenue, 3)

HP 15 Laptop
Hisense 43in Smart TV
Lenovo Tab M10


In [13]:
def high_value(rows, threshold):
    return [
        row for row in rows
        if float(row['quantity']) * float(row['unit_price_kes']) > threshold
    ]

In [14]:
# qualifying sales with a threshold of KES 50,000. 
high_sales = high_value(clean_rows, 50000)
for sale in high_sales:
    revenue = float(sale['quantity']) * float(sale['unit_price_kes'])
    print(f"{sale['product']} -- {sale['store']} -- KES {revenue:,.2f}")

print(f'Qualifying Sales: {len(high_sales)}')

Lenovo Tab M10 -- Westgate -- KES 99,996.00
Hisense 43in Smart TV -- Kisumu Mega Plaza -- KES 155,996.00
HP 15 Laptop -- Nairobi CBD -- KES 259,996.00
Tecno Spark 20 -- Nakuru Westside -- KES 53,996.00
HP 15 Laptop -- Kisumu Mega Plaza -- KES 129,998.00
Qualifying Sales: 5


In [15]:
# how many sales each store had above the threshold,
stores = {sale['store'] for sale in high_sales}

store_counts = {
    store: sum(1 for sale in high_sales if sale['store'] == store)
    for store in stores
}

print('\nHigh Value Sales by Store')
for store, count in sorted(store_counts.items()):
    print(f'{store}: {count} sales')


High Value Sales by Store
Kisumu Mega Plaza: 2 sales
Nairobi CBD: 1 sales
Nakuru Westside: 1 sales
Westgate: 1 sales


In [16]:
def mean(num_list):
    total = 0

    if not list:
        raise ValueError('Enter a non empty list')
    for item in num_list:
        total += item
    return total / len(num_list)
    
def median(num_list):
    if not num_list:
        raise ValueError('Enter a non empty list')
    sorted_list = sorted(num_list)
    list_len = len(sorted_list)

    middle = list_len // 2

    if list_len % 2 != 0:
        return sorted_list[middle]
    return (sorted_list[middle - 1] + sorted_list[middle]) / 2
    
def minimum_maximum(num_list):
    if not num_list:
        raise ValueError('Enter a non empty list')

    min_num = num_list[0]
    max_num = num_list[0]

    for num in num_list[1:]:
        if num < min_num:
            min_num = num

    for num in num_list[1:]:
        if num > min_num:
            max_num = num

    return min_num, max_num

In [18]:
transaction_revenue = [
    float(row['quantity']) * float(row['unit_price_kes'])
    for row in clean_rows
]

average_revenue = mean(transaction_revenue)
median_revenue = median(transaction_revenue)
minimum, maximum = minimum_maximum(transaction_revenue)

print(f'Mean: KES {average_revenue:,.2f}')
print(f'Median: KES {median_revenue:,.2f}')
print(f'Minimum: KES {minimum:,.2f}')
print(f'Maximum: KES {maximum:,.2f}')

Mean: KES 28,025.74
Median: KES 10,998.00
Minimum: KES 1,499.00
Maximum: KES 5,598.00


In [39]:
def verify_with_pandas(path, my_results):
    sales_df = pd.read_csv(path, keep_default_na=False)

    numeric_columns = ['quantity', 'unit_price_kes']

    for col in numeric_columns:
        if col not in sales_df.columns:
            raise ValueError('Missing Required Column')

    sales_df['quantity_cleaned'] = sales_df['quantity'].map(parse_number)
    sales_df['unit_price_cleaned'] = sales_df['unit_price_kes'].map(parse_number)

    valid = sales_df.dropna(subset=['quantity_cleaned', 'unit_price_cleaned']).copy()

    valid['revenue'] = (valid['quantity_cleaned'] * valid['unit_price_cleaned'])
    revenue = valid['revenue']

    if len(revenue) == 0:
        print('No Valid Transactions Found')
        return {}

    pandas_results = {
        'total_revenue': f'KES {revenue.sum():,.2f}',
        'mean_revenue': revenue.mean(),
        'median_revenue': revenue.median(),
        'minimum_revenue': revenue.min(),
        'maximum_revenue': revenue.max(),
        'transaction_len': len(valid)
    }

    for metric, pandas_value in pandas_results.items():
        if metric not in my_results:
            print(f'{metric}: MISMATCH')
            continue
        
        if metric == 'transaction_len':
            matches = my_results[metric] == pandas_value
        else:
            try:
                matches = math.isclose(
                    float(my_results[metric]),
                    float(pandas_value),
                )
            except (TypeError, ValueError):
                matches = False
        is_match = 'MATCH' if matches else 'MISMATCH'

        print(
            f'{metric}: {is_match} -- '
            f'Custom Value = {my_results[metric]} -- '
            f'Pandas Value = {pandas_value}'
        )
    return pandas_results

In [40]:
my_results = {
    'total_revenue': total_revenue,
    'mean_revenue': average_revenue,
    'median_revenue': median_revenue,
    'minimum_revenue': minimum,
    'maximum_revenue': maximum,
    'transaction_len': len(clean_rows),
}

verify_with_pandas('./data/electronics_sales.csv', my_results)

total_revenue: MISMATCH -- Custom Value = KES 1,317,210.00 -- Pandas Value = KES 1,317,210.00
mean_revenue: MATCH -- Custom Value = 28025.744680851065 -- Pandas Value = 28025.744680851065
median_revenue: MATCH -- Custom Value = 10998.0 -- Pandas Value = 10998.0
minimum_revenue: MATCH -- Custom Value = 1499.0 -- Pandas Value = 1499.0
maximum_revenue: MISMATCH -- Custom Value = 5598.0 -- Pandas Value = 259996.0
transaction_len: MATCH -- Custom Value = 47 -- Pandas Value = 47


{'total_revenue': 'KES 1,317,210.00',
 'mean_revenue': np.float64(28025.744680851065),
 'median_revenue': np.float64(10998.0),
 'minimum_revenue': np.float64(1499.0),
 'maximum_revenue': np.float64(259996.0),
 'transaction_len': 47}